# Lab 4 - Capstone: Design, Estimate, Defend

*SDAIA Academy · Experimentation and Causal Inference · STARTER notebook*

## Capstone brief
You are the experimentation lead for **Injaz**. Leadership faces two linked decisions:

1. **Ship the AI form-autofill feature nationally?** A randomised A/B test has run (`injaz_autofill_ab.csv`, 70,000 users). Ship threshold agreed in the design doc: lift on completion of at least **+1.5 points** with no guardrail regression.
2. **Scale the regional reminder programme?** It rolled out region by region without randomisation (`injaz_regions_panel.csv`).

Deliver a runnable notebook plus a two-page decision memo: estimand, DAG, design, estimate with interval, diagnostics, sensitivity, and a recommendation with the key assumption and what would change the call.

Rubric weights: estimand and question 15, identification and design 25, estimation and inference 25, diagnostics and sensitivity 20, communication and decision 15. A well-justified "hold" or "not identifiable" earns full marks.

In [ ]:
import sys; sys.path.insert(0, '..')   # so `causal_utils` is importable from starter/ or solution/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from causal_utils import *
plt.rcParams['figure.figsize'] = (7, 3.5)
rng = np.random.default_rng(213)
DATA = '../data'

## Part A - Frame and freeze the plan (do this BEFORE looking at outcomes)

In [ ]:
from dataclasses import dataclass, field
@dataclass
class Engagement:
    question: str; estimand: str; method: str; oec: str; guardrails: list; mde: float
    frozen: bool = False; result: dict = field(default_factory=dict)
    def freeze(self):
        assert self.method and self.oec and self.mde, 'design incomplete'; self.frozen = True
    def record(self, effect, ci, validation, recommendation):
        assert self.frozen, 'analyse only after freezing the plan'
        self.result = dict(effect=effect, ci=ci, validation=validation, recommendation=recommendation)

# TODO: fill in the engagement for the autofill decision and freeze it
autofill = Engagement(question='...', estimand='...', method='...', oec='completion_rate', guardrails=[...], mde=0.015)
autofill.freeze()

## Part B - Power check: was the test big enough for a 1.5-point MDE?

In [ ]:
ab = pd.read_csv(f'{DATA}/injaz_autofill_ab.csv'); ab['T'] = (ab.variant == 'treatment').astype(int)
# TODO: baseline rate in control, sample_size_proportions(baseline, 0.015), compare with arm sizes

## Part C - Integrity: reproduce assignment, SRM, balance

In [ ]:
# TODO: assign_variant with salt 'injaz-autofill-2026q3'; srm_check; balance_table on age, digital_literacy, pre_completion_rate

## Part D - Analysis: effect with CUPED covariate and HC3, guardrails with Bonferroni, practical-significance verdict

In [ ]:
from statsmodels.stats.multitest import multipletests
# TODO: OLS completed ~ T + pre_completion_rate with HC3 -> lift, CI
# TODO: guardrails form_error, log(time_to_complete), support_ticket with Bonferroni; flag regressions (positive effect on a 'bad' metric)
# TODO: verdict: compare the CI to the 0.015 threshold, not only to zero

## Part E - Causal evaluation of the reminder pilot (DiD + event study + placebo)
State the identifying assumption (parallel trends) and one credible threat before estimating.

In [ ]:
panel = pd.read_csv(f'{DATA}/injaz_regions_panel.csv')
# TODO: TWFE DiD with region-clustered SEs; event study (reference rel=-1); placebo timing test (6 months earlier, pre-period only)

## Part F - Decision plot and memo

In [ ]:
# TODO: decision plot with both effects and the threshold line; record the result on the Engagement; render the memo text
# The memo must contain: estimand, method, effect + CI in business units, verdict vs threshold, validation, key assumption, what would change this

## Part G - Extension (choose one)
- Heterogeneous effects (CATE) on the autofill test by device with a pre-registered note on multiplicity.
- Always-valid (mSPRT) monitoring of the autofill test: at which day could you have legitimately stopped?
- Rosenbaum-style or E-value sensitivity statement for the DiD estimate.
- Instrumental-variables estimate of filing channel on completion using `distance_km` (Lab 3 data).

## Self-audit before submission
- [ ] Plan frozen before results; fixed seed; runs top to bottom
- [ ] Estimand named; DAG or identifying assumption stated for the observational limb
- [ ] SRM + balance shown; SE type matches the design (HC3 / clustered)
- [ ] Guardrails with the right correction family; CI compared to the threshold, not only to zero
- [ ] Event study + placebo present; assumption stated as a risk
- [ ] Memo: effect + CI in business units, recommendation, what would change this